[Cell1]

In [1]:
import os
import random
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Bidirectional, LSTM, Multiply
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def set_random_seed(seed_value=42):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    print(f"Random seed set to {seed_value}")

set_random_seed(42)

2026-05-21 05:35:07.064782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-21 05:35:07.087508: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-21 05:35:07.087568: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-21 05:35:07.102371: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-21 05:35:08.090093: W tensorflow/compiler/tf

Random seed set to 42


[Cell2]

In [2]:
NUM_SAMPLES = 50000
NUM_HOURS = 24
NUM_GENS = 54

# 변환된 .npz 파일 로드
data = np.load('uc_new_data.npz')

X_demand = data['X_demand']
Y_status = data['Y_status']
Y_power  = data['Y_power']

print("=== 데이터 로드 완료 ===")
print("X_demand shape:", X_demand.shape)
print("Y_status shape:", Y_status.shape) 
print("Y_power shape:", Y_power.shape)   

=== 데이터 로드 완료 ===
X_demand shape: (50000, 24, 1)
Y_status shape: (50000, 24, 54)
Y_power shape: (50000, 24, 54)


[Cell2.5]

In [3]:
# ---------------------------------------------------------
# [새로운 cell 2.5] 정적(Static) 데이터 파싱 및 X_static 생성
# ---------------------------------------------------------
import pandas as pd
import io

# 1. 제공된 데이터를 CSV 문자열로 정의 (주석 및 빈 줄 제외하고 복사)
csv_data = """Bus,MaxProd,MinProd,IniProd,IniState,SUcap,SDcap,RampUp,RampDw,FuelCost,SlopeVarCost,InterVarCost,OMVarCost,MinTU,MinTD,SDduration,ShutdownCost,SUduration1,DownTtimeforSU1,SUcost1
gen01,4,30,5,30,5,5,5,5,5,1.00,27.08,26.55,0.00,5,4,1,0,1,4,40
gen02,6,30,5,30,5,5,5,5,5,1.00,31.30,25.85,0.00,5,4,1,0,1,4,40
gen03,8,30,5,30,5,5,5,5,5,1.00,27.90,26.10,0.00,5,4,1,0,1,4,40
gen04,10,300,150,0,-13,150,150,150,150,1.00,12.06,8.05,0.00,1,13,2,0,4,13,440
gen05,12,300,100,300,2,100,100,100,100,1.00,11.04,7.82,0.00,2,13,2,0,4,13,110
gen06,15,30,10,30,2,10,10,10,10,1.00,28.51,28.87,0.00,2,4,1,0,1,4,40
gen07,18,100,25,0,-7,25,25,25,25,1.00,14.64,12.26,0.00,3,7,1,0,2,7,50
gen08,19,30,5,30,5,5,5,5,5,1.00,30.45,28.51,0.00,5,4,1,0,1,4,40
gen09,24,30,5,30,5,5,5,5,5,1.00,26.44,26.38,0.00,5,4,1,0,1,4,40
gen10,25,300,100,300,2,100,100,100,100,1.00,11.72,7.12,0.00,2,13,2,0,4,13,100
gen11,26,350,100,300,2.5,100,100,100,100,1.00,10.11,35.59,0.00,3,13,2,0,4,13,100
gen12,27,30,8,0,-4,8,8,8,8,1.00,29.17,29.49,0.00,3,4,1,0,1,4,40
gen13,31,30,8,0,-4,8,8,8,8,1.00,30.89,27.62,0.00,3,4,1,0,1,4,40
gen14,32,100,25,0,-7,25,25,25,25,1.00,14.13,11.96,0.00,3,7,1,0,2,7,50
gen15,34,30,8,30,2.75,8,8,8,8,1.00,29.76,27.95,0.00,3,4,1,0,1,4,40
gen16,36,100,25,0,-7,25,25,25,25,1.00,16.76,11.61,0.00,3,7,1,0,2,7,50
gen17,40,30,8,0,-4,8,8,8,8,1.00,26.76,29.17,0.00,3,4,1,0,1,4,40
gen18,42,30,8,0,-4,8,8,8,8,1.00,28.10,27.25,0.00,3,4,1,0,1,4,40
gen19,46,100,25,0,-7,25,25,25,25,1.00,14.85,10.64,0.00,3,7,1,0,2,7,59
gen20,49,250,50,0,-13,50,50,50,50,1.00,10.83,30.48,0.00,4,13,2,0,4,13,100
gen21,54,250,50,50,4,50,50,50,50,1.00,10.60,30.74,0.00,4,13,2,0,4,13,100
gen22,55,100,25,0,-7,25,25,25,25,1.00,15.37,12.59,0.00,3,7,1,0,2,7,50
gen23,56,100,25,0,-7,25,25,25,25,1.00,15.19,11.87,0.00,3,7,1,0,2,7,50
gen24,59,200,50,0,-13,50,50,50,50,1.00,11.31,43.01,0.00,3,13,2,0,4,13,100
gen25,61,200,50,0,-13,50,50,50,50,1.00,13.01,39.49,0.00,3,13,2,0,4,13,100
gen26,62,100,25,0,-7,25,25,25,25,1.00,15.34,12.12,0.00,3,7,1,0,2,7,50
gen27,65,420,100,100,3.2,100,100,100,100,1.00,7.93,72.78,0.00,3,15,2,0,5,15,250
gen28,66,420,100,100,3.2,100,100,100,100,1.00,8.28,69.33,0.00,3,15,2,0,5,15,250
gen29,69,300,80,0,-13,80,80,80,80,1.00,10.99,7.91,0.00,3,13,2,0,4,13,100
gen30,70,80,30,80,1.666666667,30,30,30,30,1.00,17.83,66.83,0.00,2,7,1,0,2,7,45
gen31,72,30,10,0,-4,10,10,10,10,1.00,29.59,29.35,0.00,2,4,1,0,1,4,40
gen32,73,30,5,0,-4,5,5,5,5,1.00,30.83,27.59,0.00,5,4,1,0,1,4,40
gen33,74,20,5,0,-4,5,5,5,5,1.00,43.75,15.75,0.00,3,4,1,0,1,4,30
gen34,76,100,25,25,3,25,25,25,25,1.00,15.09,11.94,0.00,3,7,1,0,2,7,50
gen35,77,100,25,25,3,25,25,25,25,1.00,15.18,11.93,0.00,3,7,1,0,2,7,50
gen36,80,300,150,0,-13,150,150,150,150,1.00,12.28,8.22,0.00,1,13,2,0,4,13,440
gen37,82,100,25,25,3,25,25,25,25,1.00,14.73,11.73,0.00,3,7,1,0,2,7,50
gen38,85,30,10,0,-4,10,10,10,10,1.00,29.44,29.17,0.00,2,4,1,0,1,4,40
gen39,87,300,100,100,2,100,100,100,100,1.00,10.17,35.29,0.00,2,13,2,0,4,13,440
gen40,89,200,50,50,3,50,50,50,50,1.00,11.14,7.73,0.00,3,13,2,0,4,13,400
gen41,90,20,8,0,-4,8,8,8,8,1.00,44.87,15.22,0.00,2,4,1,0,1,4,30
gen42,91,50,20,40,1.5,20,20,20,20,1.00,23.87,48.05,0.00,2,4,1,0,1,4,45
gen43,92,300,100,100,2,100,100,100,100,1.00,12.65,7.28,0.00,2,13,2,0,4,13,100
gen44,99,300,100,100,2,100,100,100,100,1.00,11.17,7.92,0.00,2,13,2,0,4,13,100
gen45,100,300,100,100,2,100,100,100,100,1.00,12.10,7.41,0.00,2,13,2,0,4,13,110
gen46,103,20,8,0,-4,8,8,8,8,1.00,40.36,16.74,0.00,2,4,1,0,1,4,30
gen47,104,100,25,0,-7,25,25,25,25,1.00,14.73,11.24,0.00,3,7,1,0,2,7,50
gen48,105,100,25,0,-7,25,25,25,25,1.00,15.41,11.25,0.00,3,7,1,0,2,7,50
gen49,107,20,8,0,-4,8,8,8,8,1.00,40.55,15.77,0.00,2,4,1,0,1,4,30
gen50,110,50,25,40,1,25,25,25,25,1.00,25.28,48.88,0.00,1,4,1,0,1,4,45
gen51,111,100,25,0,-7,25,25,25,25,1.00,15.01,11.18,0.00,3,7,1,0,2,7,50
gen52,112,100,25,0,-7,25,25,25,25,1.00,15.01,10.93,0.00,3,7,1,0,2,7,50
gen53,113,100,25,0,-7,25,25,25,25,1.00,16.22,12.04,0.00,3,7,1,0,2,7,50
gen54,116,50,25,50,1,25,25,25,25,1.00,26.73,50.68,0.00,1,4,1,0,1,4,45"""

df = pd.read_csv(io.StringIO(csv_data))

LINEAR_COST_VALS = df['SlopeVarCost'].values  # 2차(또는 선형) 비용 계수
NOLOAD_COST_VALS = df['InterVarCost'].values  # 1차(또는 절편) 비용 계수
SU_COST_VALS = df['SUcost1'].values      # 기동 비용 (Start-up Cost)

MUT_VALS = df['MinTU'].values.astype(int)
MDT_VALS = df['MinTD'].values.astype(int)

# 2. 4개의 핵심 Feature 추출 및 정규화 (스케일링 지옥 방지)
# (1) Init_Status: IniState가 0보다 크면 1(켜짐), 아니면 0(꺼짐)
f_init_status = (df['IniState'] > 0).astype(float).values

# (2) Init_Power: P_max로 나누어 0~1 사이로 정규화
f_init_power = (df['IniProd'] / df['MaxProd']).values

# (3) Cost 1 (SlopeVarCost): 최대값으로 나누어 0~1 사이로 정규화
f_cost1 = (df['SlopeVarCost'] / df['SlopeVarCost'].max()).values

# (4) Cost 2 (InterVarCost): 최대값으로 나누어 0~1 사이로 정규화
f_cost2 = (df['InterVarCost'] / df['InterVarCost'].max()).values

# 3. 54개 발전기의 (54, 4) 배열 생성 후 (216,) 1차원 벡터로 평탄화
# 순서: [발전기1상태, 발전기1파워, 발전기1코스트1, 발전기1코스트2, 발전기2상태... ]
static_features_1d = np.column_stack((f_init_status, f_init_power, f_cost1, f_cost2)).flatten()

# 4. 10000개의 샘플 수에 맞게 동일한 초기값 배열을 복제
# (주의: 만약 10000개 데이터의 시작점이 매번 다르면, 이 배열을 데이터셋에서 따로 뽑아야 함!
# 현재는 118-bus 고정 스펙이므로 똑같이 복제함)
X_static = np.tile(static_features_1d, (NUM_SAMPLES, 1))

print("X_static shape:", X_static.shape) # 기대값: (10000, 216)

X_static shape: (50000, 216)


[Cell3]

In [4]:
# ---------------------------------------------------------
# [수정된 cell 3] Train, Val, Test 데이터 분할 (Static 확장 추가)
# ---------------------------------------------------------
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Power 정규화 로직 유지
Y_power_2d = Y_power.reshape(-1, NUM_GENS)
scaler_power = MinMaxScaler()
Y_power_scaled_2d = scaler_power.fit_transform(Y_power_2d)
Y_power_scaled = Y_power_scaled_2d.reshape(-1, NUM_HOURS, NUM_GENS)

# 💡 2. 핵심 해결책: X_static 길이를 전체 데이터 수(50,000개)에 맞게 강제 확장
num_samples = len(X_demand) # 50000
# X_static의 첫 번째 데이터(스펙)를 num_samples 만큼 그대로 복사해서 똑같이 만들어줌
X_static_expanded = np.tile(X_static[0], (num_samples, 1))

print(f"데이터 크기 확인 -> Demand: {len(X_demand)}, Static: {len(X_static_expanded)}")

# 3. 분할 진행 (X_static 대신 X_static_expanded 사용!)
X_demand_train, X_demand_temp, X_static_train, X_static_temp, Y_status_train, Y_status_temp, Y_power_train, Y_power_temp = train_test_split(
    X_demand, X_static_expanded, Y_status, Y_power_scaled, test_size=0.2, random_state=42
)

# Temp 20% -> Val 10%, Test 10%
X_demand_val, X_demand_test, X_static_val, X_static_test, Y_status_val, Y_status_test, Y_power_val, Y_power_test = train_test_split(
    X_demand_temp, X_static_temp, Y_status_temp, Y_power_temp, test_size=0.5, random_state=42
)

print(f"Train samples: {len(X_demand_train)}")
print(f"Val samples: {len(X_demand_val)}")
print(f"Test samples: {len(X_demand_test)}")

데이터 크기 확인 -> Demand: 50000, Static: 50000
Train samples: 40000
Val samples: 5000
Test samples: 5000


[Cell4]

In [5]:
# ---------------------------------------------------------
# [수정된 cell 4] Transformer 기반 모델 (Attention Collapse 방지)
# ---------------------------------------------------------
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Multiply, Concatenate, RepeatVector, Layer, Lambda, LayerNormalization, MultiHeadAttention, Add, Dropout, Embedding
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

# df에서 물리적 제약조건 가져오기 (cell 2.5 실행 필수)
P_MAX_VALS = df['MaxProd'].values
P_MIN_VALS = df['MinProd'].values
RU_VALS = df['RampUp'].values
RD_VALS = df['RampDw'].values

def ste_binarize(x):
    return x + tf.stop_gradient(tf.round(x) - x)

class DynamicHybridPILossLayer(Layer):
    def __init__(self, p_max, p_min, ru, rd, **kwargs):
        super(DynamicHybridPILossLayer, self).__init__(**kwargs)
        self.p_max = tf.constant(p_max, dtype=tf.float32)
        self.p_min = tf.constant(p_min, dtype=tf.float32)
        self.ru = tf.constant(ru, dtype=tf.float32)
        self.rd = tf.constant(rd, dtype=tf.float32)

        self.w_mismatch = tf.Variable(1.0, trainable=False, dtype=tf.float32)
        self.w_capacity = tf.Variable(1.0, trainable=False, dtype=tf.float32)
        self.w_ramp = tf.Variable(1.0, trainable=False, dtype=tf.float32)

    def call(self, inputs):
        demand_mw_input, status_ste, out_power = inputs
        pred_power_mw = out_power * self.p_max
        demand_mw = tf.squeeze(demand_mw_input, axis=-1)

        pred_total = tf.reduce_sum(pred_power_mw, axis=-1)
        mismatch_mw = tf.reduce_mean(tf.abs(pred_total - demand_mw))

        upper_violation = tf.maximum(pred_power_mw - self.p_max, 0.0) * status_ste
        lower_violation = tf.maximum(self.p_min - pred_power_mw, 0.0) * status_ste
        capacity_mw = tf.reduce_mean(tf.reduce_sum(upper_violation + lower_violation, axis=-1))

        delta_p = pred_power_mw[:, 1:, :] - pred_power_mw[:, :-1, :]
        stay_on_mask = status_ste[:, 1:, :] * status_ste[:, :-1, :]
        ramp_up_violation = tf.maximum(delta_p - self.ru, 0.0) * stay_on_mask
        ramp_down_violation = tf.maximum(-delta_p - self.rd, 0.0) * stay_on_mask
        ramp_mw = tf.reduce_mean(tf.reduce_sum(ramp_up_violation + ramp_down_violation, axis=-1))

        total_penalty = (self.w_mismatch * mismatch_mw +
                         self.w_capacity * capacity_mw +
                         self.w_ramp * ramp_mw) / 10000.0

        self.add_loss(total_penalty)
        return out_power

    def compute_output_shape(self, input_shape):
        return input_shape[2]

class PositionalEmbedding(Layer):
    def __init__(self, sequence_length, output_dim, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        self.pos_emb = Embedding(input_dim=sequence_length, output_dim=output_dim)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        return inputs + self.pos_emb(positions)

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = LayerNormalization(epsilon=1e-6)(inputs)

    # 💡 [핵심 2번] use_causal_mask=True 적용!
    # 이제 모델은 t 시간의 출력을 낼 때 t+1 시간 이후의 데이터를 훔쳐볼 수 없습니다.
    x = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x, x, use_causal_mask=True) # <- 이 파라미터 하나가 마법을 만듭니다.

    x = Add()([x, inputs])

    res = x
    x = LayerNormalization(epsilon=1e-6)(x)
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    return Add()([x, res])

def build_transformer_uc_model(num_hours=24, num_gens=54, num_static_features=4):
    input_dynamic = Input(shape=(num_hours, 1), name='demand_input')
    static_dim = num_gens * num_static_features
    input_static = Input(shape=(static_dim,), name='static_initial_input')

    static_encoded = Dense(64, activation='relu', name='static_encoder')(input_static)
    static_repeated = RepeatVector(num_hours, name='static_repeater')(static_encoded)
    merged_input = Concatenate(axis=-1, name='feature_fusion')([input_dynamic, static_repeated])

    # 💡 [핵심 1번] 차원(Dimension) 2배 벌크업! (128 -> 256)
    projected_input = Dense(256, activation='relu', name='feature_projection')(merged_input)
    x = PositionalEmbedding(sequence_length=num_hours, output_dim=256, name='positional_encoding')(projected_input)

    # 💡 [핵심 1번] 헤드 갯수 2배(8개), Feed Forward 망 2배(512) 확장!
    x = transformer_encoder(x, head_size=256, num_heads=8, ff_dim=512, dropout=0.1)
    x = transformer_encoder(x, head_size=256, num_heads=8, ff_dim=512, dropout=0.1)

    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x) # 서서히 줄여줌

    # ------------------ (출력부 및 레이어는 기존과 100% 동일) ------------------
    out_status = Dense(num_gens, activation='sigmoid', name='out_status')(x)
    status_ste = Lambda(ste_binarize, name='status_ste')(out_status)

    raw_power = Dense(num_gens, activation='sigmoid', name='raw_power')(x)
    out_power_raw = Multiply(name='out_power_raw')([raw_power, status_ste])

    out_power = DynamicHybridPILossLayer(
        P_MAX_VALS, P_MIN_VALS, RU_VALS, RD_VALS, name='out_power_dynamic'
    )([input_dynamic, status_ste, out_power_raw])

    model = Model(inputs=[input_dynamic, input_static], outputs=[out_status, out_power])
    return model

NUM_STATIC_FEATURES = 4
model = build_transformer_uc_model(NUM_HOURS, NUM_GENS, NUM_STATIC_FEATURES)
model.summary()

2026-05-21 05:35:15.603375: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 869 MB memory:  -> device: 0, name: CUDA GPU, pci bus id: 0000:90:00.0, compute capability: 8.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ static_initial_inp… │ (None, 216)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_encoder      │ (None, 64)        │     13,888 │ static_initial_i… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ demand_input        │ (None, 24, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static_repeater     │ (None, 24, 64)    │          0 │ static_encoder[0… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_fusion      │ (None, 24, 65)    │          0 │ demand_input[0][… │
│ (Concatenate)       │                   │            │ static_repeater[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_projection  │ (None, 24, 256)   │     16,896 │ feature_fusion[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 24, 256)   │      6,144 │ feature_projecti… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 24, 256)   │        512 │ positional_encod… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 24, 256)   │  2,103,552 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 256)   │          0 │ multi_head_atten… │
│                     │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 24, 512)   │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 24, 512)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24, 256)   │    131,328 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 24, 256)   │          0 │ dense_1[0][0],    │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 24, 256)   │  2,103,552 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 24, 256)   │          0 │ multi_head_atten

 Total params: 4,884,524 (18.63 MB)

 Trainable params: 4,884,524 (18.63 MB)

 Non-trainable params: 0 (0.00 B)

[Cell5]

In [6]:
import tensorflow as tf

# ---------------------------------------------------------
# 1. 54개 발전기의 P_min, P_max 하드코딩 (단위: MW)
# ---------------------------------------------------------
P_min_vals = [
    5.0, 5.0, 5.0, 150.0, 100.0, 10.0, 25.0, 5.0, 5.0, 100.0, 100.0, 8.0, 8.0, 25.0, 8.0,
    25.0, 8.0, 8.0, 25.0, 50.0, 50.0, 25.0, 25.0, 50.0, 50.0, 25.0, 100.0, 100.0, 80.0,
    30.0, 10.0, 5.0, 5.0, 25.0, 25.0, 150.0, 25.0, 10.0, 100.0, 50.0, 8.0, 20.0, 100.0,
    100.0, 100.0, 8.0, 25.0, 25.0, 8.0, 25.0, 25.0, 25.0, 25.0, 25.0
]

P_max_vals = [
    30.0, 30.0, 30.0, 300.0, 300.0, 30.0, 100.0, 30.0, 30.0, 300.0, 350.0, 30.0, 30.0,
    100.0, 30.0, 100.0, 30.0, 30.0, 100.0, 250.0, 250.0, 100.0, 100.0, 200.0, 200.0,
    100.0, 420.0, 420.0, 300.0, 80.0, 30.0, 30.0, 20.0, 100.0, 100.0, 300.0, 100.0, 30.0,
    300.0, 200.0, 20.0, 50.0, 300.0, 300.0, 300.0, 20.0, 100.0, 100.0, 20.0, 50.0, 100.0,
    100.0, 100.0, 50.0
]

# ---------------------------------------------------------
# 2. 54개 발전기의 Ramp-Up (RU), Ramp-Down (RD) 하드코딩 (단위: MW/h)
# ---------------------------------------------------------
# (주의: Julia에서는 Ramp 값을 3배 늘렸지만, AI 학습 평가를 위해 원래 스펙을 로드합니다.
# 만약 완화된 3배 스펙을 원하시면 나중에 이 텐서에 * 3.0 을 해주면 됩니다.)
RU_vals = [
    5.0, 5.0, 5.0, 150.0, 100.0, 10.0, 25.0, 5.0, 5.0, 100.0, 100.0, 8.0, 8.0, 25.0, 8.0,
    25.0, 8.0, 8.0, 25.0, 50.0, 50.0, 25.0, 25.0, 50.0, 50.0, 25.0, 100.0, 100.0, 80.0,
    30.0, 10.0, 5.0, 5.0, 25.0, 25.0, 150.0, 25.0, 10.0, 100.0, 50.0, 8.0, 20.0, 100.0,
    100.0, 100.0, 8.0, 25.0, 25.0, 8.0, 25.0, 25.0, 25.0, 25.0, 25.0
]

RD_vals = [
    5.0, 5.0, 5.0, 150.0, 100.0, 10.0, 25.0, 5.0, 5.0, 100.0, 100.0, 8.0, 8.0, 25.0, 8.0,
    25.0, 8.0, 8.0, 25.0, 50.0, 50.0, 25.0, 25.0, 50.0, 50.0, 25.0, 100.0, 100.0, 80.0,
    30.0, 10.0, 5.0, 5.0, 25.0, 25.0, 150.0, 25.0, 10.0, 100.0, 50.0, 8.0, 20.0, 100.0,
    100.0, 100.0, 8.0, 25.0, 25.0, 8.0, 25.0, 25.0, 25.0, 25.0, 25.0
]

# ---------------------------------------------------------
# 3. TensorFlow 상수(Tensor)로 변환 (Loss 함수 연산용)
# ---------------------------------------------------------
p_min_tensor = tf.constant(P_min_vals, dtype=tf.float32)
p_max_tensor = tf.constant(P_max_vals, dtype=tf.float32)
ru_tensor = tf.constant(RU_vals, dtype=tf.float32)
rd_tensor = tf.constant(RD_vals, dtype=tf.float32)

# 이전 코드 호환용: P_MAX_tensor가 역정규화에 쓰이고 있었으므로 매핑
P_MAX_tensor = p_max_tensor

print("✅ 54개 발전기의 물리적 제약조건 텐서가 메모리에 정상적으로 등록되었습니다!")

✅ 54개 발전기의 물리적 제약조건 텐서가 메모리에 정상적으로 등록되었습니다!


: 

[Cell6]

In [ ]:
# ---------------------------------------------------------
# [수정된 cell 6] Phase 1: 발전기 Status(On/Off) 집중 학습
# ---------------------------------------------------------
print("=== Phase 1: 발전기 Status(On/Off) 집중 학습 시작 ===")

import tensorflow as tf

# 💡 'out_power'를 모두 'out_power_dynamic'으로 변경
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'out_status': 'binary_crossentropy',
        'out_power_dynamic': 'mse'
    },
    loss_weights={
        'out_status': 1.0,
        'out_power_dynamic': 0.0  # Phase 1은 Status 학습에 집중
    },
    metrics={'out_status': ['accuracy'], 'out_power_dynamic': ['mae']}
)

es_phase1 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    mode='min'
)

history_phase1 = model.fit(
    x=[X_demand_train, X_static_train],
    # 💡 정답지 딕셔너리의 키도 변경
    y={'out_status': Y_status_train, 'out_power_dynamic': Y_power_train},
    validation_data=(
        [X_demand_val, X_static_val],
        {'out_status': Y_status_val, 'out_power_dynamic': Y_power_val}
    ),
    epochs=50,  # 기존 에포크 설정 유지
    batch_size=64,
    callbacks=[es_phase1],
    verbose=1
)
print("=== Phase 1 학습 종료 ===")

=== Phase 1: 발전기 Status(On/Off) 집중 학습 시작 ===
Epoch 1/50


I0000 00:00:1779341724.989099   17355 service.cc:145] XLA service 0x7fe42000b2e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779341724.989139   17355 service.cc:153]   StreamExecutor device (0): CUDA GPU, Compute Capability 8.0
2026-05-21 05:35:25.148068: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-21 05:35:25.723304: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1779341732.795329   17421 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_112', 16 bytes spill stores, 24 bytes spill loads

I0000 00:00:1779341732.852818   17422 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_112', 48 bytes spill stores, 48 bytes spill loads

I0000 00:00:1779341733.0788

[Cell7]

In [ ]:
# ---------------------------------------------------------
# [수정된 cell 7] Phase 2: 코사인 Warm-up + Status 절대 방어막 커리큘럼
# ---------------------------------------------------------
import tensorflow as tf
import tensorflow.keras.backend as K
import math

# 💡 1. 커리큘럼 콜백 (기존의 Status 방어막 유지)
class CurriculumCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        layer = self.model.get_layer('out_power_dynamic')

        if epoch < 3:
            K.set_value(layer.w_mismatch, 0.5)
            K.set_value(layer.w_capacity, 0.5)
            K.set_value(layer.w_ramp, 0.1)
            phase_name = "Phase A (마이크로 워밍업)"
        elif epoch < 20:
            K.set_value(layer.w_mismatch, 6.0)
            K.set_value(layer.w_capacity, 1.0)
            K.set_value(layer.w_ramp, 0.1)
            phase_name = "Phase B (자유도 부여 & 수요 추종)"
        elif epoch < 40:
            K.set_value(layer.w_mismatch, 10.0)
            K.set_value(layer.w_capacity, 3.0)
            K.set_value(layer.w_ramp, 0.5)
            phase_name = "Phase C (용량 제약 안정화)"
        else:
            K.set_value(layer.w_mismatch, 15.0)
            K.set_value(layer.w_capacity, 3.0)
            K.set_value(layer.w_ramp, 1.0)
            phase_name = "Phase D (최종 파레토 수렴)"

        print(f"\n🚀 [Epoch {epoch+1} / {phase_name}] 가중치 -> Mis: {K.get_value(layer.w_mismatch)}, Cap: {K.get_value(layer.w_capacity)}, Ramp: {K.get_value(layer.w_ramp)}")

# 💡 2. 코사인 Warm-up 스케줄러 함수 정의
def cosine_warmup_scheduler(epoch, lr):
    warmup_epochs = 10     # 처음 10 에포크 동안 워밍업
    total_epochs = 60      # 전체 에포크 수
    max_lr = 5e-4          # 워밍업 완료 시 최대 속도 (0.0005)
    min_lr = 1e-6          # 학습 종료 시 최소 속도 (0.000001)

    if epoch < warmup_epochs:
        # 워밍업 구간: 0 근처에서 max_lr까지 부드럽게 상승
        return max_lr * ((epoch + 1) / warmup_epochs)
    else:
        # 코사인 하강 구간: max_lr에서 min_lr로 부드럽게 감소
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

print("=== Phase 2: 데이터 펌핑 + 코사인 Warm-up 학습 시작 ===")

# 컴파일 (초기 학습률은 스케줄러가 덮어씌우므로 기본값 세팅)
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss={
        'out_status': 'binary_crossentropy',
        'out_power_dynamic': 'mse'
    },
    loss_weights={
        'out_status': 10.0,            # 🛡️ Status 절대 방어막
        'out_power_dynamic': 1.0
    },
    metrics={'out_status': ['accuracy'], 'out_power_dynamic': ['mae']}
)

# 💡 기존 ReduceLROnPlateau 대신 커스텀 스케줄러 콜백 장착
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(cosine_warmup_scheduler, verbose=1)
curriculum_cb = CurriculumCallback()

history_phase2 = model.fit(
    x=[X_demand_train, X_static_train],
    y={'out_status': Y_status_train, 'out_power_dynamic': Y_power_train},
    validation_data=(
        [X_demand_val, X_static_val],
        {'out_status': Y_status_val, 'out_power_dynamic': Y_power_val}
    ),
    epochs=60,
    batch_size=64, # 데이터가 많아졌으므로 배치 사이즈를 128로 늘려도 좋습니다.
    callbacks=[curriculum_cb, lr_scheduler],
    verbose=1
)
print("=== Phase 2 학습 종료 ===")

[Cell8]

In [ ]:
# ---------------------------------------------------------
# [수정된 cell 8] Phase 1 & Phase 2 통합 커리큘럼 학습 곡선 (Learning Curve)
# ---------------------------------------------------------
import matplotlib.pyplot as plt

# 1. Phase 1과 Phase 2의 History 데이터 병합
train_loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

train_acc = history_phase1.history['out_status_accuracy'] + history_phase2.history['out_status_accuracy']
val_acc = history_phase1.history['val_out_status_accuracy'] + history_phase2.history['val_out_status_accuracy']

train_mae = history_phase1.history['out_power_dynamic_mae'] + history_phase2.history['out_power_dynamic_mae']
val_mae = history_phase1.history['val_out_power_dynamic_mae'] + history_phase2.history['val_out_power_dynamic_mae']

# 2. 각 Phase별 에포크 경계선 계산 (EarlyStopping 조기 종료 고려)
p1_len = len(history_phase1.history['loss'])
p2_len = len(history_phase2.history['loss'])
total_epochs = p1_len + p2_len

# 커리큘럼 기준점
phase_A_end = p1_len + min(15, p2_len)
phase_B_end = p1_len + min(40, p2_len)

# 3. 배경색 및 경계선을 그려주는 도우미 함수
def add_curriculum_backgrounds(ax):
    # 배경색 칠하기
    ax.axvspan(0, p1_len, color='lightgray', alpha=0.3, label='Phase 1 (Status 집중)')
    if p2_len > 0:
        ax.axvspan(p1_len, phase_A_end, color='lightblue', alpha=0.3, label='Phase 2A (자유도 부여)')
    if p2_len > 15:
        ax.axvspan(phase_A_end, phase_B_end, color='lightgreen', alpha=0.3, label='Phase 2B (용량 안정화)')
    if p2_len > 40:
        ax.axvspan(phase_B_end, total_epochs, color='lightcoral', alpha=0.3, label='Phase 2C (Mismatch 올인)')

    # 경계선 그리기
    ax.axvline(p1_len, color='black', linestyle='-', linewidth=1.5) # Phase 1 -> 2 경계
    ax.axvline(phase_A_end, color='gray', linestyle='--', linewidth=1)
    ax.axvline(phase_B_end, color='gray', linestyle='--', linewidth=1)

# 4. 3개의 서브플롯 그리기
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (1) Total Loss 그래프
axes[0].plot(train_loss, label='Train Total Loss', color='blue', linewidth=2)
axes[0].plot(val_loss, label='Val Total Loss', color='orange', linestyle='--', linewidth=2)
axes[0].set_title('Curriculum Learning: Total Loss', fontsize=14)
axes[0].set_ylabel('Loss (MSE + Dynamic Penalties)')
axes[0].set_yscale('log') # Loss 스케일이 클 수 있으므로 Log 스케일 적용

# (2) Status Accuracy 그래프
axes[1].plot(train_acc, label='Train Status Acc', color='purple', linewidth=2)
axes[1].plot(val_acc, label='Val Status Acc', color='magenta', linestyle='--', linewidth=2)
axes[1].set_title('Curriculum Learning: Status Accuracy', fontsize=14)
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0.5, 1.05])

# (3) Power MAE 그래프
axes[2].plot(train_mae, label='Train Power MAE', color='green', linewidth=2)
axes[2].plot(val_mae, label='Val Power MAE', color='red', linestyle='--', linewidth=2)
axes[2].set_title('Curriculum Learning: Power MAE', fontsize=14)
axes[2].set_ylabel('Mean Absolute Error (Scaled)')

# 공통 설정 적용
for ax in axes:
    add_curriculum_backgrounds(ax)
    ax.set_xlabel('Total Epochs')
    ax.grid(True, alpha=0.4)
    # 범례가 겹치지 않게 깔끔하게 배치
    handles, labels = ax.get_legend_handles_labels()
    # 중복 라벨 제거
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

[Cell9]

In [ ]:
# ---------------------------------------------------------
# [수정된 cell 9] Test Set 평가 (Multi-Input 지원)
# ---------------------------------------------------------
import numpy as np

print("=== Test Set 물리적 제약 기반 최종 평가 시작 ===")

# 💡 수정 포인트: predict() 함수에 [동적 데이터, 정적 데이터] 리스트 전달
pred_status, pred_power_scaled = model.predict([X_demand_test, X_static_test])

# ---------------------------------------------------------
# Status 이진화 및 Power 역정규화
# ---------------------------------------------------------
pred_status_bin = (pred_status > 0.5).astype(int)

pred_power_2d = pred_power_scaled.reshape(-1, NUM_GENS)
pred_power_mw_2d = scaler_power.inverse_transform(pred_power_2d)
pred_power_mw = pred_power_mw_2d.reshape(-1, NUM_HOURS, NUM_GENS)

true_power_2d = Y_power_test.reshape(-1, NUM_GENS)
true_power_mw_2d = scaler_power.inverse_transform(true_power_2d)
true_power_mw = true_power_mw_2d.reshape(-1, NUM_HOURS, NUM_GENS)

SYS_CAPACITY = 7220.0
P_MIN_np = np.array(P_min_vals)
P_MAX_np = np.array(P_max_vals)
RU_np = np.array(RU_vals)
RD_np = np.array(RD_vals)

# 💡 수정 포인트: X_test 대신 X_demand_test 사용
X_test_mw = X_demand_test.squeeze(-1)

# ---------------------------------------------------------
# ML 성능 및 수급 불균형 평가 (Mismatch)
# ---------------------------------------------------------
status_accuracy = np.mean(pred_status_bin == Y_status_test) * 100
power_mae = np.mean(np.abs(pred_power_mw - true_power_mw))

total_pred_power_mw = np.sum(pred_power_mw, axis=-1)
mismatch_mae = np.mean(np.abs(total_pred_power_mw - X_test_mw))

mean_actual_demand_mw = np.mean(X_test_mw)
relative_error_percent = (mismatch_mae / mean_actual_demand_mw) * 100

# ---------------------------------------------------------
# 물리적 제약 위반 (Violation) 검사
# ---------------------------------------------------------
ghost_power_mw = np.maximum(pred_power_mw, 0.0) * (1 - pred_status_bin)
mean_ghost_violation = np.mean(np.sum(ghost_power_mw, axis=-1))
ghost_relative_percent = (mean_ghost_violation / mean_actual_demand_mw) * 100

upper_violation = np.maximum(pred_power_mw - P_MAX_np, 0.0) * pred_status_bin
lower_violation = np.maximum(P_MIN_np - pred_power_mw, 0.0) * pred_status_bin
mean_capacity_violation = np.mean(np.sum(upper_violation + lower_violation, axis=-1))
capacity_relative_percent = (mean_capacity_violation / mean_actual_demand_mw) * 100

delta_p = pred_power_mw[:, 1:, :] - pred_power_mw[:, :-1, :]
stay_on_mask = pred_status_bin[:, 1:, :] * pred_status_bin[:, :-1, :]

ramp_up_violation = np.maximum(delta_p - RU_np, 0.0) * stay_on_mask
ramp_down_violation = np.maximum(-delta_p - RD_np, 0.0) * stay_on_mask
mean_ramp_violation = np.mean(np.sum(ramp_up_violation + ramp_down_violation, axis=-1))
ramp_relative_percent = (mean_ramp_violation / mean_actual_demand_mw) * 100

# ---------------------------------------------------------
# 결과 출력
# ---------------------------------------------------------
print("\n" + "="*50)
print("🎯 E2E UC 모델 (Multi-Input) 물리-AI 성능 평가 지표")
print("="*50)
print(f"1. 발전기 기동(On/Off) 정확도  : {status_accuracy:.2f}%")
print(f"2. 개별 발전량 평균 오차 (MAE) : {power_mae:.2f}MW")
print(f"3. 전력 수급 불균형 (Mismatch) : {mismatch_mae:.2f}MW ({relative_error_percent:.2f}%)")
print(f"4. 평균 1시간 수요 규모        : {mean_actual_demand_mw:.2f}MW")
print(f"5. 수급 불균형 상대 오차(NMAE) : {relative_error_percent:.2f}%")
print("-" * 50)
print("[물리적 제약 위반]")
print(f"6. 유령 발전 위반 (Ghost Power): {mean_ghost_violation:>6.2f}MW ({ghost_relative_percent:.2f}%)")
print(f"7. 용량 제약 위반 (Capacity)   : {mean_capacity_violation:>6.2f}MW ({capacity_relative_percent:.2f}%)")
print(f"8. 증감발률 위반 (Ramp Rate)   : {mean_ramp_violation:>6.2f}MW ({ramp_relative_percent:.2f}%)")
print("="*50)

In [ ]:
# ---------------------------------------------------------
# [수정된 cell 9] Test Set 평가 (Multi-Input 지원)
# ---------------------------------------------------------
import numpy as np

print("=== Test Set 물리적 제약 기반 최종 평가 시작 ===")

# 💡 수정 포인트: predict() 함수에 [동적 데이터, 정적 데이터] 리스트 전달
pred_status, pred_power_scaled = model.predict([X_demand_test, X_static_test])

# ---------------------------------------------------------
# Status 이진화 및 Power 역정규화
# ---------------------------------------------------------
pred_status_bin = (pred_status > 0.5).astype(int)

pred_power_2d = pred_power_scaled.reshape(-1, NUM_GENS)
pred_power_mw_2d = scaler_power.inverse_transform(pred_power_2d)
pred_power_mw = pred_power_mw_2d.reshape(-1, NUM_HOURS, NUM_GENS)

true_power_2d = Y_power_test.reshape(-1, NUM_GENS)
true_power_mw_2d = scaler_power.inverse_transform(true_power_2d)
true_power_mw = true_power_mw_2d.reshape(-1, NUM_HOURS, NUM_GENS)

SYS_CAPACITY = 7220.0
P_MIN_np = np.array(P_min_vals)
P_MAX_np = np.array(P_max_vals)
RU_np = np.array(RU_vals)
RD_np = np.array(RD_vals)

# 💡 수정 포인트: X_test 대신 X_demand_test 사용
X_test_mw = X_demand_test.squeeze(-1)

# ---------------------------------------------------------
# ML 성능 및 수급 불균형 평가 (Mismatch)
# ---------------------------------------------------------
status_accuracy = np.mean(pred_status_bin == Y_status_test) * 100
power_mae = np.mean(np.abs(pred_power_mw - true_power_mw))

total_pred_power_mw = np.sum(pred_power_mw, axis=-1)
mismatch_mae = np.mean(np.abs(total_pred_power_mw - X_test_mw))

mean_actual_demand_mw = np.mean(X_test_mw)
relative_error_percent = (mismatch_mae / mean_actual_demand_mw) * 100

# ---------------------------------------------------------
# 물리적 제약 위반 (Violation) 검사
# ---------------------------------------------------------
ghost_power_mw = np.maximum(pred_power_mw, 0.0) * (1 - pred_status_bin)
mean_ghost_violation = np.mean(np.sum(ghost_power_mw, axis=-1))
ghost_relative_percent = (mean_ghost_violation / mean_actual_demand_mw) * 100

upper_violation = np.maximum(pred_power_mw - P_MAX_np, 0.0) * pred_status_bin
lower_violation = np.maximum(P_MIN_np - pred_power_mw, 0.0) * pred_status_bin
mean_capacity_violation = np.mean(np.sum(upper_violation + lower_violation, axis=-1))
capacity_relative_percent = (mean_capacity_violation / mean_actual_demand_mw) * 100

delta_p = pred_power_mw[:, 1:, :] - pred_power_mw[:, :-1, :]
stay_on_mask = pred_status_bin[:, 1:, :] * pred_status_bin[:, :-1, :]

ramp_up_violation = np.maximum(delta_p - RU_np, 0.0) * stay_on_mask
ramp_down_violation = np.maximum(-delta_p - RD_np, 0.0) * stay_on_mask
mean_ramp_violation = np.mean(np.sum(ramp_up_violation + ramp_down_violation, axis=-1))
ramp_relative_percent = (mean_ramp_violation / mean_actual_demand_mw) * 100

# =========================================================
# 💡 [비용 평가] AI 예측 비용 vs CPLEX 정답지 비용 비교 분석
# =========================================================
import numpy as np

def calculate_total_cost_from_numpy(status_arr, power_mw_arr, linear_cost, noload_cost, su_cost):
    """
    status_arr: (samples, 24, 54) 형태의 기동 상태 (0 or 1)
    power_mw_arr: (samples, 24, 54) 형태의 실제 발전량 (MW)
    linear_cost, noload_cost, su_cost: 데이터셋 스펙에 맞춘 올바른 비용 계수 배열
    """
    samples, hours, gens = status_arr.shape
    total_cost_per_sample = []
    
    for s in range(samples):
        sample_cost = 0
        status = status_arr[s]
        power = power_mw_arr[s]
        
        for t in range(hours):
            # 1. 💡 운전 비용 = (선형 비용 * P) + (무부하 유지 비용 * Status)
            op_cost = np.sum(linear_cost * power[t] + noload_cost * status[t])
            
            # 2. 기동 비용 (Startup Cost) = 이전 시간에 꺼져있다가(0) 켜지면(1) 부과
            if t == 0:
                # 0시 당일 첫 기동 측정
                su_events = np.maximum(status[t] - 0, 0)
            else:
                su_events = np.maximum(status[t] - status[t-1], 0)
            
            st_cost = np.sum(su_cost * su_events)
            
            sample_cost += (op_cost + st_cost)
            
        total_cost_per_sample.append(sample_cost)
        
    return np.mean(total_cost_per_sample) # 평균 1일 비용 리턴

# 1. 정답지 (Ground Truth) 발전량을 0~1 스케일에서 실제 MW 단위로 역정규화
Y_power_mw_test = Y_power_test * P_MAX_VALS

# 2. 비용 계산 실행 
# (이전에 Cell 2.5에서 정의한 LINEAR_COST_VALS, NOLOAD_COST_VALS를 사용합니다)
ai_avg_cost = calculate_total_cost_from_numpy(pred_status_bin, pred_power_mw, LINEAR_COST_VALS, NOLOAD_COST_VALS, SU_COST_VALS)
cplex_avg_cost = calculate_total_cost_from_numpy(Y_status_test, Y_power_mw_test, LINEAR_COST_VALS, NOLOAD_COST_VALS, SU_COST_VALS)

# 3. 결과 출력
cost_diff = ai_avg_cost - cplex_avg_cost
cost_diff_percent = (cost_diff / cplex_avg_cost) * 100

def calculate_overall_mut_mdt_metrics(all_pred_status, mut_vals, mdt_vals):
    """
    all_pred_status: shape (num_samples, 24, 54) - 테스트 셋 전체의 예측 Status
    """
    num_samples, num_hours, num_gens = all_pred_status.shape
    
    total_mut_violation_hours = 0
    total_mdt_violation_hours = 0
    total_startup_events = 0
    total_shutdown_events = 0
    
    for s in range(num_samples):
        for g in range(num_gens):
            mut = mut_vals[g]
            mdt = mdt_vals[g]
            status = all_pred_status[s, :, g]
            
            for t in range(1, num_hours):
                # 기동 (Startup)
                if status[t-1] == 0 and status[t] == 1:
                    total_startup_events += 1
                    check_len = min(mut, num_hours - t)
                    actual_on = np.sum(status[t : t+check_len])
                    if actual_on < check_len:
                        total_mut_violation_hours += (check_len - actual_on)
                        
                # 정지 (Shutdown)
                elif status[t-1] == 1 and status[t] == 0:
                    total_shutdown_events += 1
                    check_len = min(mdt, num_hours - t)
                    actual_on = np.sum(status[t : t+check_len])
                    if actual_on > 0:
                        total_mdt_violation_hours += actual_on

    # 지표 계산
    avg_mut_violation_per_sample = total_mut_violation_hours / num_samples
    avg_mdt_violation_per_sample = total_mdt_violation_hours / num_samples
    
    print(f"{'='*30}\n[Overall Performance & Physical Constraints]\n{'='*30}")
    print(f"Generator Status (On/Off) Accuracy: {status_accuracy:.2f}%")
    print(f"Mean Absolute Error (MAE) of Power Generation: {power_mae:.2f} MW")
    print(f"Power Balance Violation (Mismatch): {mismatch_mae:.2f} MW ({relative_error_percent:.2f}% NMAE)")
    print(f"Ghost Power Violation: {mean_ghost_violation:.2f} MW ({ghost_relative_percent:.2f}%)")
    print(f"Capacity Violation: {mean_capacity_violation:.2f} MW ({capacity_relative_percent:.2f}%)")
    print(f"Ramp Rate Violation: {mean_ramp_violation:.2f} MW ({ramp_relative_percent:.2f}%)")

    print(f"{'='*30}\n[Economic Evaluation]\n{'='*30}")
    print(f"Average Daily Cost (CPLEX Ground Truth): ${cplex_avg_cost:,.2f}")
    print(f"Average Daily Cost (AI Prediction): ${ai_avg_cost:,.2f}")
    if cost_diff > 0:
        print(f"Cost Difference: The AI model's scheduling is ${cost_diff:,.2f} (+{cost_diff_percent:.2f}%) more expensive than the CPLEX optimal solution.")
    else:
        print(f"Cost Difference: The AI model's scheduling is ${abs(cost_diff):,.2f} ({cost_diff_percent:.2f}%) cheaper than the CPLEX optimal solution. (Note: This slight cost reduction is a theoretical trade-off achieved through minor constraint violations).")

    print(f"{'='*30}\n[Temporal Constraints]\n{'='*30}")
    print(f"Average Minimum Up Time (MUT) Violation: {avg_mut_violation_per_sample:.2f} hours/sample")
    print(f"Average Minimum Down Time (MDT) Violation: {avg_mdt_violation_per_sample:.2f} hours/sample")

# 실행 예시
calculate_overall_mut_mdt_metrics(pred_status_bin, MUT_VALS, MDT_VALS)

[Cell10]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("=== 개별 샘플 심층 시각화 분석 (Multi-Input 반영) ===")

sample_power_errors = np.mean(np.abs(pred_power_mw - true_power_mw), axis=(1, 2))

best_idx = np.argmin(sample_power_errors)
worst_idx = np.argmax(sample_power_errors)
median_error = np.median(sample_power_errors)
avg_idx = np.argmin(np.abs(sample_power_errors - median_error))

def plot_sample_analysis(idx, title_prefix="Sample"):
    fig = plt.figure(figsize=(18, 14))

    t_demand = X_test_mw[idx]
    t_true_power_total = np.sum(true_power_mw[idx], axis=1)
    t_pred_power_total = np.sum(pred_power_mw[idx], axis=1)

    status_diff = Y_status_test[idx] - pred_status_bin[idx]
    power_diff = true_power_mw[idx] - pred_power_mw[idx]

    ax1 = plt.subplot(3, 1, 1)
    ax1.plot(t_demand, label='Actual Demand', color='black', linestyle='--', linewidth=2)
    ax1.plot(t_true_power_total, label='True Total Power (Label)', color='blue', linewidth=2, marker='o')
    ax1.plot(t_pred_power_total, label='Pred Total Power (AI)', color='orange', linewidth=2, marker='x')

    ax1.set_title(f"[{title_prefix}] System Total Power vs Demand (Index: {idx})", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Power (MW)", fontsize=12)
    ax1.set_xticks(range(NUM_HOURS))
    ax1.set_xticklabels([f"{h+1}h" for h in range(NUM_HOURS)])
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.5)

    ax2 = plt.subplot(3, 1, 2)
    sns.heatmap(status_diff.T, cmap='coolwarm', center=0, cbar_kws={'ticks': [-1, 0, 1]}, ax=ax2)
    ax2.set_title("Status Mismatch Heatmap (Generators vs Hours)", fontsize=14, fontweight='bold')
    ax2.set_ylabel("Generator ID", fontsize=12)
    ax2.set_xlabel("Hour", fontsize=12)
    cbar = ax2.collections[0].colorbar
    cbar.set_ticklabels(['False ON\n(Ghost)', 'Match\n(Correct)', 'False OFF\n(Miss)'])

    ax3 = plt.subplot(3, 1, 3)
    sns.heatmap(power_diff.T, cmap='RdBu', center=0, ax=ax3, robust=True)
    ax3.set_title("Power Error Heatmap [True - Pred] (MW)", fontsize=14, fontweight='bold')
    ax3.set_ylabel("Generator ID", fontsize=12)
    ax3.set_xlabel("Hour", fontsize=12)

    plt.tight_layout()
    plt.show()

plot_sample_analysis(best_idx, title_prefix="BEST Sample")
plot_sample_analysis(avg_idx, title_prefix="AVERAGE Sample")
plot_sample_analysis(worst_idx, title_prefix="WORST Sample")